# FRAG-SED Benchmark: End-to-End LSED Official (`w=infty`, ACL'25) on Colab GPU

Runs **Stage 1 (`Qwen/Qwen3-8B` batch summarization)** + **Stage 2 (`all-MiniLM-L6-v2` + 3-layer HGCN Poincare MLP training on `M_00` & `Oracle-K` / `HISEvent Est-K` clustering)** across full streams:
1. **`event2012_full`** (`M_00` 20,254 + `M_01..M_21` 48,587 rows; automatically reuses existing `event2012_dev` & `colab_outputs` responses!)
2. **`event2018_full`** (`M_00` 14,328 + `M_01..M_16` 50,188 rows; automatically reuses existing `event2018_dev` responses!)

Settings match `trackA_sug_sed/baselines_official.py` verbatim (`Qwen/Qwen3-8B`, `temp=0.0`, `enable_thinking=False`, `max_tokens=256`; `all-MiniLM-L6-v2` `max_length=256`; 3-layer HGCN Poincare MLP `hidden_dim=64`, `lr=0.01`, `tan_reg=0.02`, `weight_decay=1e-4`, `epochs=200`, `patience=30`, `KMeans n_init=10`).
Select **Runtime -> Change runtime type -> GPU** (L4/A100 recommended), then **Run all**.

In [ ]:
from google.colab import drive
import json, os, torch

drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/FRAG_SED_Experiments'

FULL_RUNS = [
    {
        'name': 'event2012_full',
        'dataset': 'Event2012',
        'prompt_file': 'lsed_M01_M21_prompts.jsonl',
        'response_file': 'lsed_M01_M21_responses.jsonl',
        'm0_response_candidates': [
            os.path.join(BASE_DIR, 'event2012_dev', 'colab_outputs', 'lsed_M00_responses.jsonl'),
            os.path.join(BASE_DIR, 'colab_outputs', 'lsed_M00_responses.jsonl'),
        ],
        'seed_response_files': [
            os.path.join(BASE_DIR, 'event2012_dev', 'colab_outputs', 'lsed_M02_M05_responses.jsonl'),
            os.path.join(BASE_DIR, 'colab_outputs', 'lsed_M01_M05_responses.jsonl'),
        ],
    },
    {
        'name': 'event2018_full',
        'dataset': 'Event2018',
        'prompt_file': 'lsed_M01_M16_prompts.jsonl',
        'response_file': 'lsed_M01_M16_responses.jsonl',
        'm0_response_candidates': [
            os.path.join(BASE_DIR, 'event2018_dev', 'colab_outputs', 'lsed_M00_responses.jsonl'),
        ],
        'seed_response_files': [
            os.path.join(BASE_DIR, 'event2018_dev', 'colab_outputs', 'lsed_M02_M05_responses.jsonl'),
        ],
    },
]

assert torch.cuda.is_available(), 'No GPU detected! Please switch Colab Runtime to GPU (T4/L4/A100).'
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name}, VRAM: {vram_gb:.1f} GB')

def _count_done(path):
    ids = set()
    if os.path.exists(path):
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    try:
                        ids.add(json.loads(line)['request_id'])
                    except Exception:
                        pass
    return ids

any_llm_todo = False
for cfg in FULL_RUNS:
    in_d = os.path.join(BASE_DIR, cfg['name'], 'colab_inputs')
    out_d = os.path.join(BASE_DIR, cfg['name'], 'colab_outputs')
    os.makedirs(out_d, exist_ok=True)
    ip = os.path.join(in_d, cfg['prompt_file'])
    op = os.path.join(out_d, cfg['response_file'])
    assert os.path.exists(ip), f'Missing prompt file: {ip}'
    assert os.path.exists(os.path.join(in_d, 'lsed_eval_meta.npz')), f'Missing meta npz in {in_d}'
    target_ids = {json.loads(l)['request_id'] for l in open(ip, 'r', encoding='utf-8') if l.strip()}
    covered = _count_done(op)
    for sp in cfg['seed_response_files']:
        covered |= _count_done(sp)
    rem = len(target_ids - covered)
    if rem > 0:
        any_llm_todo = True
    m0_found = next((p for p in cfg['m0_response_candidates'] if os.path.exists(p)), None)
    print(f"{cfg['name']}: prompts={len(target_ids)}, covered={len(target_ids & covered)}, remaining={rem}, m0_file={os.path.relpath(m0_found, BASE_DIR) if m0_found else 'MISSING'}")

In [ ]:
# Cell 2: Install & load Qwen/Qwen3-8B only when prompts remain (vLLM single-process on >=22GB; Transformers bf16 fallback; 4-bit on T4)
import gc

MODEL_ID = 'Qwen/Qwen3-8B'
USE_VLLM = any_llm_todo and (vram_gb >= 22.0)
BATCH_SIZE = 32 if vram_gb >= 22.0 else 24
llm = None
model = None
tokenizer = None

if not any_llm_todo:
    print('All full-stream LLM responses are already complete; skipping Qwen3-8B load.')
else:
    if USE_VLLM:
        os.environ['VLLM_ENABLE_V1_MULTIPROCESSING'] = '0'
        !pip install -q vllm>=0.8.5
        from vllm import LLM, SamplingParams
        from transformers import AutoTokenizer
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, padding_side='left')
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        try:
            llm = LLM(
                model=MODEL_ID,
                dtype='bfloat16',
                max_model_len=1024,
                gpu_memory_utilization=0.85,
                enforce_eager=True,
                max_num_seqs=64,
            )
            print('Loaded vLLM bf16 backend (V1 single-process, enforce_eager=True).')
        except Exception as vllm_err:
            print(f'vLLM startup failed ({type(vllm_err).__name__}: {vllm_err}); falling back to Transformers.')
            USE_VLLM = False
            gc.collect()
            torch.cuda.empty_cache()

    if not USE_VLLM:
        !pip install -q transformers>=4.51.0 accelerate bitsandbytes
        from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, padding_side='left')
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        if vram_gb >= 22.0:
            model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto')
            print('Loaded Transformers bf16 fallback backend.')
        else:
            bnb_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type='nf4')
            model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_cfg, device_map='auto')
            print('Loaded 4-bit Transformers backend for T4.')
        model.eval()

In [ ]:
# Cell 3: Multi-seed JSONL prompt runner with resume & exact token counts
import time
from tqdm.auto import tqdm

def format_prompt(rec):
    msgs = []
    if rec.get('sys_msg'):
        msgs.append({'role': 'system', 'content': rec['sys_msg']})
    msgs.append({'role': 'user', 'content': rec['user_msg']})
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)

def run_jsonl_task(in_path, out_path, max_new_tokens, batch_size=32, seed_responses_paths=None):
    with open(in_path, 'r', encoding='utf-8') as f:
        items = [json.loads(line) for line in f if line.strip()]
    target_ids = {r['request_id'] for r in items}

    done_ids = set()
    if os.path.exists(out_path):
        with open(out_path, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    try:
                        done_ids.add(json.loads(line)['request_id'])
                    except Exception:
                        pass

    for sp_path in (seed_responses_paths or []):
        if not sp_path or not os.path.exists(sp_path):
            continue
        reused = 0
        with open(sp_path, 'r', encoding='utf-8') as sf, open(out_path, 'a', encoding='utf-8') as fout:
            for line in sf:
                if not line.strip():
                    continue
                try:
                    obj = json.loads(line)
                except Exception:
                    continue
                rid = obj.get('request_id')
                if rid in target_ids and rid not in done_ids and isinstance(obj.get('text'), str) and obj.get('text').strip() and obj.get('prompt_tokens', 0) > 0:
                    fout.write(json.dumps(obj, ensure_ascii=False) + '\n')
                    done_ids.add(rid)
                    reused += 1
        if reused > 0:
            print(f'Reused {reused} responses from {os.path.relpath(sp_path, BASE_DIR)}')

    todo = [r for r in items if r['request_id'] not in done_ids]
    print(f'{os.path.relpath(in_path, BASE_DIR)}: total={len(items)}, already_done={len(done_ids)}, remaining={len(todo)}')
    if not todo:
        return

    t0 = time.time()
    with open(out_path, 'a', encoding='utf-8') as fout:
        if USE_VLLM:
            chunk_sz = 512
            sp = SamplingParams(temperature=0.0, max_tokens=max_new_tokens)
            for s in tqdm(range(0, len(todo), chunk_sz), desc=os.path.basename(in_path)):
                chunk = todo[s : s + chunk_sz]
                prompts = [format_prompt(r) for r in chunk]
                outs = llm.generate(prompts, sp, use_tqdm=False)
                for r, out in zip(chunk, outs):
                    txt = out.outputs[0].text.strip()
                    pt = len(out.prompt_token_ids)
                    ct = len(out.outputs[0].token_ids)
                    fout.write(json.dumps({
                        'request_id': r['request_id'],
                        'block_id': r['block_id'],
                        'text': txt,
                        'prompt_tokens': pt,
                        'completion_tokens': ct,
                    }, ensure_ascii=False) + '\n')
                fout.flush()
        else:
            for s in tqdm(range(0, len(todo), batch_size), desc=os.path.basename(in_path)):
                chunk = todo[s : s + batch_size]
                prompts = [format_prompt(r) for r in chunk]
                enc = tokenizer(prompts, return_tensors='pt', padding=True, truncation=True, max_length=896).to(model.device)
                in_len = enc['input_ids'].shape[1]
                with torch.no_grad():
                    gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id)
                for bi, r in enumerate(chunk):
                    pt = int(enc['attention_mask'][bi].sum().item())
                    out_ids = gen[bi, in_len:]
                    eos_pos = (out_ids == tokenizer.eos_token_id).nonzero(as_tuple=True)[0]
                    if len(eos_pos) > 0:
                        out_ids = out_ids[: int(eos_pos[0].item()) + 1]
                    ct = int(len(out_ids))
                    txt = tokenizer.decode(out_ids, skip_special_tokens=True).strip()
                    fout.write(json.dumps({
                        'request_id': r['request_id'],
                        'block_id': r['block_id'],
                        'text': txt,
                        'prompt_tokens': pt,
                        'completion_tokens': ct,
                    }, ensure_ascii=False) + '\n')
                if (s // batch_size) % 10 == 0:
                    fout.flush()
            fout.flush()
    print(f'Finished {in_path} in {time.time() - t0:.1f}s -> {out_path}')

In [ ]:
# Cell 4: Run Stage-1 LLM summarization across both full streams, then free Qwen3-8B VRAM
for cfg in FULL_RUNS:
    in_p = os.path.join(BASE_DIR, cfg['name'], 'colab_inputs', cfg['prompt_file'])
    out_p = os.path.join(BASE_DIR, cfg['name'], 'colab_outputs', cfg['response_file'])
    print(f"=== Summarizing {cfg['name']} ===")
    run_jsonl_task(in_p, out_p, max_new_tokens=256, batch_size=BATCH_SIZE, seed_responses_paths=cfg['seed_response_files'])

del llm, model, tokenizer
gc.collect()
torch.cuda.empty_cache()
print('Released Qwen3-8B GPU memory cleanly.')

In [ ]:
# Cell 5: Encode LSED English summaries with sentence-transformers/all-MiniLM-L6-v2 (mean-pool + L2 norm, max_length=256)
import numpy as np
from transformers import AutoModel, AutoTokenizer

SBERT_MODEL_ID = 'sentence-transformers/all-MiniLM-L6-v2'
sbert_tok = AutoTokenizer.from_pretrained(SBERT_MODEL_ID)
sbert_mdl = AutoModel.from_pretrained(SBERT_MODEL_ID).eval().to('cuda')

def encode_texts_minilm(texts, batch_size=256, max_length=256):
    if not texts:
        return np.zeros((0, 384), dtype=np.float32)
    chunks = []
    with torch.no_grad():
        for s in range(0, len(texts), batch_size):
            batch = texts[s : s + batch_size]
            enc = sbert_tok(batch, padding=True, truncation=True, max_length=max_length, return_tensors='pt').to('cuda')
            out = sbert_mdl(**enc)
            mask = enc['attention_mask'].unsqueeze(-1).float()
            pooled = (out.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
            pooled = torch.nn.functional.normalize(pooled, p=2, dim=-1)
            chunks.append(pooled.cpu().numpy().astype(np.float32))
    return np.vstack(chunks)

print('Loaded SBERT encoder:', SBERT_MODEL_ID)

In [ ]:
# Cell 6: Verbatim LSED Official 3-layer HGCN Poincare MLP + OLE 2D TIME (exact parity with trackA_sug_sed/baselines_official.py)
import copy, math
import torch.nn as nn
import torch.nn.functional as F
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_mutual_info_score, adjusted_rand_score, normalized_mutual_info_score

def _ole_time_vec(timestamps):
    ole_offset_sec = 2209161600.0
    valid = ~np.isnan(timestamps)
    out = np.zeros((len(timestamps), 2), dtype=np.float32)
    if np.any(valid):
        sec_since_ole = timestamps[valid] + ole_offset_sec
        t_days = np.floor(sec_since_ole / 86400.0)
        t_secs = sec_since_ole - t_days * 86400.0
        out[valid, 0] = (t_days / 100000.0).astype(np.float32)
        out[valid, 1] = (t_secs / 86400.0).astype(np.float32)
    return out

def _poincare_proj(x, c=1.0, eps=1e-5):
    max_norm = (1.0 / math.sqrt(c)) - eps
    norm = torch.norm(x, p=2, dim=-1, keepdim=True)
    return torch.where(norm > max_norm, x * (max_norm / norm.clamp(min=1e-9)), x)

def _poincare_exp0(v, c=1.0):
    sqrt_c = math.sqrt(c)
    norm = torch.norm(v, p=2, dim=-1, keepdim=True).clamp(min=1e-7)
    return _poincare_proj((torch.tanh(sqrt_c * norm) / (sqrt_c * norm)) * v, c=c)

def _poincare_log0(y, c=1.0):
    sqrt_c = math.sqrt(c)
    max_norm = (1.0 / sqrt_c) - 1e-5
    norm = torch.norm(y, p=2, dim=-1, keepdim=True).clamp(min=1e-7, max=max_norm)
    return (torch.atanh(sqrt_c * norm) / (sqrt_c * norm)) * y

def _poincare_mobius_add(x, y, c=1.0):
    x2 = (x * x).sum(dim=-1, keepdim=True)
    y2 = (y * y).sum(dim=-1, keepdim=True)
    xy = (x * y).sum(dim=-1, keepdim=True)
    num = (1.0 + 2.0 * c * xy + c * y2) * x + (1.0 - c * x2) * y
    denom = (1.0 + 2.0 * c * xy + (c * c) * x2 * y2).clamp(min=1e-7)
    return _poincare_proj(num / denom, c=c)

def _build_lsed_hgcn_model(in_dim, hidden_dim, n_cls, num_layers=3, c=1.0, dropout=0.0):
    class _HypLinear(nn.Module):
        def __init__(self, d_in, d_out):
            super().__init__()
            self.weight = nn.Parameter(torch.empty(d_out, d_in))
            self.bias = nn.Parameter(torch.zeros(d_out))
            nn.init.xavier_uniform_(self.weight)
            self.drop = nn.Dropout(p=dropout) if dropout > 0.0 else nn.Identity()
        def forward(self, x):
            tan_x = self.drop(_poincare_log0(x, c=c))
            res = _poincare_exp0(F.linear(tan_x, self.weight), c=c)
            return _poincare_mobius_add(res, _poincare_exp0(self.bias, c=c), c=c)

    class _HypAct(nn.Module):
        def __init__(self):
            super().__init__()
            self.act = nn.ReLU()
        def forward(self, x):
            return _poincare_exp0(self.act(_poincare_log0(x, c=c)), c=c)

    class _LSEDHGCNPoincareEncoder(nn.Module):
        def __init__(self):
            super().__init__()
            self.c = float(c)
            blocks = []
            d_cur = in_dim
            for li in range(max(1, int(num_layers))):
                blocks.append(_HypLinear(d_cur, hidden_dim))
                if li < num_layers - 1:
                    blocks.append(_HypAct())
                d_cur = hidden_dim
            self.hyp_layers = nn.Sequential(*blocks)
            self.cls = nn.Linear(hidden_dim, n_cls)
        def encode_hyperbolic(self, x_euc):
            return _poincare_proj(self.hyp_layers(_poincare_exp0(x_euc, c=self.c)), c=self.c)
        def forward(self, x_euc):
            return self.cls(_poincare_log0(self.encode_hyperbolic(x_euc), c=self.c))

    return _LSEDHGCNPoincareEncoder()

def train_hgcn_on_m0(labels_m0, ts_m0, m0_texts, ckpt_path, hidden_dim=64, num_layers=3, curvature=1.0, lr=0.01, epochs=200, weight_decay=1e-4, tan_reg=0.02, patience=30, seed=42):
    _, y_inv = np.unique(labels_m0, return_inverse=True)
    n_cls = int(y_inv.max()) + 1
    in_dim = 386
    device = 'cuda'
    model = _build_lsed_hgcn_model(in_dim, hidden_dim, n_cls, num_layers=num_layers, c=curvature).to(device)
    if os.path.exists(ckpt_path):
        loaded = torch.load(ckpt_path, map_location=device, weights_only=True)
        if isinstance(loaded, dict) and loaded.get('arch') == 'hgcn_poincare_v2' and loaded.get('m0_summarized') and int(loaded.get('n_cls', -1)) == n_cls:
            model.load_state_dict(loaded['state_dict'])
            model.eval()
            diag = {k: loaded[k] for k in ('val_acc', 'test_acc', 'mean_hyp_norm', 'sat_pct')}
            print(f'Loaded cached HGCN checkpoint {os.path.basename(ckpt_path)}: {diag}')
            return model, diag

    emb_m0 = encode_texts_minilm(m0_texts, batch_size=256, max_length=256)
    feat = np.hstack([emb_m0, _ole_time_vec(ts_m0)]).astype(np.float32)
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    n = len(feat)
    perm = rng.permutation(n)
    n_tr, n_va = max(1, int(0.7 * n)), max(1, int(0.1 * n))
    idx_tr, idx_va, idx_te = perm[:n_tr], perm[n_tr : min(n, n_tr + n_va)], perm[min(n, n_tr + n_va):]
    x_tr, y_tr = torch.from_numpy(feat[idx_tr]).to(device), torch.from_numpy(y_inv[idx_tr]).to(device)
    x_va, y_va = torch.from_numpy(feat[idx_va]).to(device), torch.from_numpy(y_inv[idx_va]).to(device)
    x_te, y_te = torch.from_numpy(feat[idx_te]).to(device), torch.from_numpy(y_inv[idx_te]).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss()
    best_loss, best_state, no_improve = float('inf'), copy.deepcopy(model.state_dict()), 0
    for _ep in range(epochs):
        model.train()
        opt.zero_grad()
        e_hyp = model.encode_hyperbolic(x_tr)
        tan_e = _poincare_log0(e_hyp, c=curvature)
        loss = loss_fn(model.cls(tan_e), y_tr) + float(tan_reg) * tan_e.pow(2).sum(dim=-1).mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step()
        model.eval()
        with torch.no_grad():
            va_loss = float(loss_fn(model(x_va), y_va).item())
        if va_loss < best_loss - 1e-4:
            best_loss, best_state, no_improve = va_loss, copy.deepcopy(model.state_dict()), 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        va_acc = round(float((model(x_va).argmax(dim=-1) == y_va).float().mean().item()), 4)
        te_acc = round(float((model(x_te).argmax(dim=-1) == y_te).float().mean().item()), 4)
        hyp_norms = torch.norm(model.encode_hyperbolic(x_va), p=2, dim=-1)
        mean_hyp_norm = round(float(hyp_norms.mean().item()), 4)
        sat_pct = round(float((hyp_norms > (0.999 / math.sqrt(curvature))).float().mean().item() * 100.0), 2)
    diag = {'val_acc': va_acc, 'test_acc': te_acc, 'mean_hyp_norm': mean_hyp_norm, 'sat_pct': sat_pct}
    torch.save({'arch': 'hgcn_poincare_v2', 'state_dict': best_state, 'in_dim': in_dim, 'hidden_dim': hidden_dim, 'num_layers': num_layers, 'n_cls': n_cls, 'curvature': float(curvature), 'm0_summarized': True, **diag}, ckpt_path)
    print(f'[LSED HGCN M_00] {diag}')
    return model, diag

In [ ]:
# Cell 7: Train M_00 HGCN, evaluate all blocks (M_01..M_T) under Oracle-K & Est-K, and save lsed_full_results.json
def _load_resp_map(path):
    out = {}
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                obj = json.loads(line)
                rid = obj.get('request_id')
                if rid:
                    out[str(rid)] = obj
    return out

def _agg(pb, bids):
    return {
        'mean': {
            'ari': round(float(np.mean([pb[f'M_{b:02d}']['ari'] for b in bids])), 2),
            'nmi': round(float(np.mean([pb[f'M_{b:02d}']['nmi'] for b in bids])), 2),
            'ami': round(float(np.mean([pb[f'M_{b:02d}']['ami'] for b in bids])), 2),
        },
        'total_calls': sum(int(pb[f'M_{b:02d}']['calls']) for b in bids),
        'total_tokens': sum(int(pb[f'M_{b:02d}']['tokens']) for b in bids),
        'total_parse_errors': sum(int(pb[f'M_{b:02d}']['parse_errors']) for b in bids),
    }

for cfg in FULL_RUNS:
    print(f"\n================ Evaluating {cfg['name']} ({cfg['dataset']}) ================")
    in_d = os.path.join(BASE_DIR, cfg['name'], 'colab_inputs')
    out_d = os.path.join(BASE_DIR, cfg['name'], 'colab_outputs')
    meta = np.load(os.path.join(in_d, 'lsed_eval_meta.npz'), allow_pickle=False)
    block_ids = [int(b) for b in meta['block_ids'].tolist()]

    m0_path = next(p for p in cfg['m0_response_candidates'] if os.path.exists(p))
    m0_map = _load_resp_map(m0_path)
    labels_m0 = meta['labels_M00']
    ts_m0 = meta['timestamps_M00']
    m0_texts, m0_pt, m0_ct = [], 0, 0
    for i in range(len(labels_m0)):
        rec = m0_map.get(f'lsed_M00_n{i}', {})
        txt = str(rec.get('text', '')).strip() or 'empty'
        m0_texts.append(txt)
        m0_pt += int(rec.get('prompt_tokens', 0) or 0)
        m0_ct += int(rec.get('completion_tokens', 0) or 0)

    ckpt_path = os.path.join(out_d, 'lsed_m00_hgcn_poincare.pt')
    hyp_model, hgcn_diag = train_hgcn_on_m0(labels_m0, ts_m0, m0_texts, ckpt_path)

    resp_map = _load_resp_map(os.path.join(out_d, cfg['response_file']))
    pb_oracle, pb_est = {}, {}
    for bid in block_ids:
        y_true = meta[f'labels_M{bid:02d}']
        ts_b = meta[f'timestamps_M{bid:02d}']
        k_oracle = len(set(y_true.tolist()))
        k_est = int(meta[f'k_est_M{bid:02d}'][0])
        n = len(y_true)
        b_texts, pt_b, ct_b, errs_b = [], 0, 0, 0
        for i in range(n):
            rec = resp_map.get(f'lsed_M{bid:02d}_n{i}')
            if rec is None:
                raise RuntimeError(f'Missing response lsed_M{bid:02d}_n{i}!')
            txt = str(rec.get('text', '')).strip()
            if not txt:
                errs_b += 1
                txt = 'empty'
            b_texts.append(txt)
            pt_b += int(rec.get('prompt_tokens', 0) or 0)
            ct_b += int(rec.get('completion_tokens', 0) or 0)
        emb_b = encode_texts_minilm(b_texts, batch_size=256, max_length=256)
        feat_b = np.hstack([emb_b, _ole_time_vec(ts_b)]).astype(np.float32)
        with torch.no_grad():
            hyp_e = hyp_model.encode_hyperbolic(torch.from_numpy(feat_b).to('cuda')).cpu().numpy().astype(np.float64)
        mean_hyp_norm = round(float(np.linalg.norm(hyp_e, axis=1).mean()), 4)
        lbl_o = KMeans(n_clusters=max(1, min(k_oracle, n)), random_state=42, n_init=10).fit_predict(hyp_e)
        lbl_e = KMeans(n_clusters=max(1, min(k_est, n)), random_state=42, n_init=10).fit_predict(hyp_e)
        common = {
            'calls': n, 'rewritten_ok': n - errs_b, 'tokens': pt_b + ct_b,
            'prompt_tokens': pt_b, 'completion_tokens': ct_b, 'shown_posts': n,
            'parse_errors': errs_b, 'mean_hyp_norm': mean_hyp_norm, 'open_set_window': 'infty',
        }
        mo = {'ari': round(float(adjusted_rand_score(y_true, lbl_o)*100), 2), 'nmi': round(float(normalized_mutual_info_score(y_true, lbl_o)*100), 2), 'ami': round(float(adjusted_mutual_info_score(y_true, lbl_o)*100), 2), **common}
        me = {'ari': round(float(adjusted_rand_score(y_true, lbl_e)*100), 2), 'nmi': round(float(normalized_mutual_info_score(y_true, lbl_e)*100), 2), 'ami': round(float(adjusted_mutual_info_score(y_true, lbl_e)*100), 2), **common}
        pb_oracle[f'M_{bid:02d}'] = mo
        pb_est[f'M_{bid:02d}'] = me
        print(f"  M_{bid:02d} (N={n:4d}, K*={k_oracle:2d}, Kest={k_est:2d}): OracleK={mo['ari']:.2f}/{mo['nmi']:.2f}/{mo['ami']:.2f}  EstK={me['ari']:.2f}/{me['nmi']:.2f}/{me['ami']:.2f}  tok={pt_b+ct_b:,}")

    m0_cost = {'calls': len(labels_m0), 'tokens': m0_pt + m0_ct, 'prompt_tokens': m0_pt, 'completion_tokens': m0_ct}
    dev_bids = [2, 3, 4, 5]
    held_bids = [b for b in block_ids if b >= 6]
    out_obj = {
        'dataset': cfg['dataset'],
        'm0_summary_cost': m0_cost,
        'hgcn_diag': hgcn_diag,
        'lsed_unbounded_oracleK': {**_agg(pb_oracle, block_ids), 'm0_summary_cost': m0_cost, 'per_block': pb_oracle},
        'lsed_unbounded_estK': {**_agg(pb_est, block_ids), 'm0_summary_cost': m0_cost, 'per_block': pb_est},
        'dev_m02_m05': {'oracleK': _agg(pb_oracle, dev_bids), 'estK': _agg(pb_est, dev_bids)},
        'held_out_m06_plus': {'oracleK': _agg(pb_oracle, held_bids), 'estK': _agg(pb_est, held_bids)},
    }
    res_file = os.path.join(out_d, 'lsed_full_results.json')
    with open(res_file, 'w', encoding='utf-8') as f:
        json.dump(out_obj, f, indent=2, ensure_ascii=False)
    print(f"--> Full Stream (M_01..M_{max(block_ids):02d}) Oracle-K: {out_obj['lsed_unbounded_oracleK']['mean']} | Est-K: {out_obj['lsed_unbounded_estK']['mean']} | Tokens={out_obj['lsed_unbounded_oracleK']['total_tokens']:,} (+M_00 {m0_cost['tokens']:,})")
    print(f"--> Dev (M_02..M_05)         Oracle-K: {out_obj['dev_m02_m05']['oracleK']['mean']} | Est-K: {out_obj['dev_m02_m05']['estK']['mean']}")
    print(f"--> Held-Out (M_06..M_{max(block_ids):02d})    Oracle-K: {out_obj['held_out_m06_plus']['oracleK']['mean']} | Est-K: {out_obj['held_out_m06_plus']['estK']['mean']}")
    print('Saved ->', res_file)